# AIDD8122 Final Project — XPert-HG-only
**Reproduction and Evaluation of an HG-Only XPert Variant for Drug-Induced Transcriptomic Response Prediction**  
Macao Polytechnic University · P2600216 · Qiu Dian

**Project repository:** https://github.com/valerieee11/AIDD8122-XPert-HG-Only  
**Upstream XPert:** https://github.com/GSanShui/XPert  
**Public data:** https://zenodo.org/records/17182939

This cleaned notebook documents the *course adaptation*, **not** a full reproduction of the original XPert paper. It uses a 1,400-sample L1000 subset; 978 genes; precomputed drug HG embeddings; UniMol disabled; a simplified three-term MSE training objective; and PyTorch native attention instead of FlashAttention. It reproduces the **workflow used in the previously completed experiment**, with independent validation and testing. The official `train_xpert.py` entry point is **not** invoked: the custom training loop appears below.

> **Status:** Notebook reviewed and reorganized from the actual completed work. Its code has been statically checked, **but not executed end-to-end in a fresh Colab session**. Saved results in `results/` are the evidence from the earlier completed run; exact metrics are not guaranteed on every rerun.

## 1 · Runtime and dependency setup
Use **Google Colab** with a compatible NVIDIA GPU (previous run: Tesla T4), and use your own **fork** as the code source. No paid GPU is needed just to inspect saved CSV results, but full training requires a functioning GPU runtime.

**Run the following installation cell once, then restart the Colab session if packages were changed.** After restarting, continue at the repository setup cell. Avoid also running the original notebook's unpinned `pip install anndata`, because it previously pulled an incompatible pandas release.

Tested environment in the original run: Python 3.13.16, PyTorch 2.14.1+cu130, pandas 2.2.3, AnnData 0.12.7. Colab's environment can change; GPU availability is not guaranteed.

In [ ]:
%pip install -q "pandas==2.2.3" "anndata==0.12.7" "bottleneck>=1.5.0" "PyYAML" "h5py" "scipy" "matplotlib"

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib
import numpy as np
import pandas as pd
import torch
import anndata as ad
import yaml

root = Path('/content/XPert')
if not root.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/valerieee11/AIDD8122-XPert-HG-Only.git',
        str(root)
    ], check=True)
os.chdir(root)

# Ensure the repository's datasets package takes priority over a third-party package.
(root / 'datasets/__init__.py').touch(exist_ok=True)
if str(root) in sys.path:
    sys.path.remove(str(root))
sys.path.insert(0, str(root))
sys.modules.pop('datasets', None)
importlib.invalidate_caches()

print('Project:', root)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Pandas:', pd.__version__)
print('AnnData:', importlib.metadata.version('anndata') if hasattr(importlib, 'metadata') else ad.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

required = [
    'models/model_XPert.py', 'models/model_utils.py', 'utils.py',
    'datasets/MyDataset.py', 'configs/config_l1000_course_hg.yaml',
    'HG_data/saved_embedding/HG_drug_embeddings.npy',
    'processed_data/PPI_gene_vector_128d.npy'
]
missing = [p for p in required if not (root / p).exists()]
if missing:
    raise FileNotFoundError(f'Required repository files missing: {missing}')

# Adapted fork should already contain native attention; do not patch upstream again.
attention_src = (root / 'models/model_utils.py').read_text(encoding='utf-8')
assert 'scaled_dot_product_attention' in attention_src,     'The cloned model_utils.py does not appear to contain the T4 attention adaptation.'
print('Adapted repository check: OK')

## 2 · Obtain the official L1000 SDST data
The public Zenodo archive is approximately 1.6 GB and contains several datasets. This notebook extracts **only** `l1000_sdst_78453.h5ad`. Ensure adequate Colab disk space. If the local archive was removed when the runtime reset, rerun this section; saved model results on Google Drive remain separate.

In [ ]:
from pathlib import Path
import zipfile, shutil, subprocess

archive = Path('/content/processed_data.zip')
full_path = root / 'processed_data/l1000_sdst_78453.h5ad'
full_path.parent.mkdir(parents=True, exist_ok=True)

if not full_path.exists():
    if not archive.exists():
        subprocess.run([
            'wget', '-c', '-O', str(archive),
            'https://zenodo.org/records/17182939/files/processed_data.zip?download=1'
        ], check=True)
    if not zipfile.is_zipfile(archive):
        raise RuntimeError('The downloaded data archive is not a valid ZIP file.')
    with zipfile.ZipFile(archive) as zf:
        matches = [name for name in zf.namelist()
                   if name.endswith('l1000_sdst_78453.h5ad')]
        if len(matches) != 1:
            raise RuntimeError(f'Expected 1 dataset file, found: {matches}')
        with zf.open(matches[0]) as src, full_path.open('wb') as dst:
            shutil.copyfileobj(src, dst)

full = ad.read_h5ad(full_path, backed='r')
print('Full data shape:', full.shape)
print('Original split counts:', full.obs['split_1'].value_counts())
assert full.shape == (78453, 978)
assert 'X_ctl' in full.obsm
full.file.close()

## 3 · Construct the deterministic 1,400-sample subset
The original paper's `split_1` train/test boundary is respected. We sample 1,200 records from its train partition, then assign 1,000 train + 200 validation; 200 further records come only from its original test partition. This is **not** a strict drug-disjoint or cold-drug split.

Random seed for data sampling: **2026**. Preserve the fixed test subset; do not use test metrics to tune hyperparameters.

In [ ]:
from scipy import sparse
rng = np.random.default_rng(2026)
full = ad.read_h5ad(full_path, backed='r')
original = full.obs['split_1'].astype(str).to_numpy()
train_pool = np.where(original == 'train')[0]
test_pool = np.where(original == 'test')[0]
assert len(train_pool) >= 1200 and len(test_pool) >= 200

train_valid = rng.choice(train_pool, size=1200, replace=False)
train_idx, valid_idx = train_valid[:1000], train_valid[1000:]
test_idx = rng.choice(test_pool, size=200, replace=False)
selected = np.sort(np.concatenate([train_idx, valid_idx, test_idx]))
mini = full[selected, :].to_memory()
full.file.close()
labels = np.full(len(selected), 'test', dtype=object)
labels[np.isin(selected, train_idx)] = 'train'
labels[np.isin(selected, valid_idx)] = 'valid'
mini.obs['split_course_large'] = labels
assert mini.shape == (1400, 978)
assert mini.obs['split_course_large'].value_counts().to_dict() == {
    'train': 1000, 'valid': 200, 'test': 200
}
mini_path = root / 'processed_data/l1000_sdst_course_1400.h5ad'
mini.write_h5ad(mini_path)
print('Saved dataset:', mini_path)
print('Splits:', mini.obs['split_course_large'].value_counts())
print('Unique drug IDs:', mini.obs['pert_idx'].nunique())

## 4 · HG-only drug feature interface and configuration
The course variant uses **real, precomputed drug knowledge-graph embeddings** (`HG_drug_embeddings.npy`). The generated `(8981, 2, 514)` zero array below is solely an **interface placeholder** for the UniMol path; it is **not** a learned molecular representation. `wo_unimol=True` is explicitly enabled when constructing the model.

In [ ]:
hg_path = root / 'HG_data/saved_embedding/HG_drug_embeddings.npy'
hg = np.load(hg_path, mmap_mode='r')
max_drug_idx = int(mini.obs['pert_idx'].max())
assert hg.ndim == 2 and hg.shape[1] == 256
assert max_drug_idx < hg.shape[0]

placeholder = np.zeros((hg.shape[0], 2, 514), dtype=np.float32)
placeholder[:, 0, 0] = 1
placeholder_path = root / 'processed_data/course_hg_only_large_placeholder.npy'
np.save(placeholder_path, placeholder)

base_config = root / 'configs/config_l1000_course_hg.yaml'
with base_config.open(encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['dataset']['drug_unimol_path'] = 'processed_data/course_hg_only_large_placeholder.npy'
cfg['dataset']['max_atom_size'] = 2
cfg['train']['batch_size'] = 1
cfg['train']['num_epochs'] = 10
cfg_path = root / 'configs/config_l1000_course_large.yaml'
with cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('HG:', hg.shape, '| Placeholder:', placeholder.shape)
print('Config:', cfg_path)

## 5 · No-change baseline
A simple comparator sets the treated expression equal to the untreated/control expression. This baseline predicts **zero perturbation-induced change**. Its Pearson correlation for perturbation deltas is undefined, not zero.

In [ ]:
def dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)

test_mask = mini.obs['split_course_large'].astype(str).eq('test').to_numpy()
y_test = dense(mini.X[test_mask]).astype(np.float32)
c_test = dense(mini.obsm['X_ctl'][test_mask]).astype(np.float32)
assert y_test.shape == c_test.shape == (200, 978)
base_mse = float(np.mean((y_test - c_test) ** 2))
base_rmse = float(np.sqrt(base_mse))
base_mae = float(np.mean(np.abs(y_test - c_test)))
print('No-change baseline:', dict(MSE=base_mse, RMSE=base_rmse, MAE=base_mae))
assert abs(base_mse - 0.519923210144043) < 1e-4,     'The baseline differs from the recorded experiment; inspect the data version and split.'

## 6 · XPert-HG-only model setup
The model is imported from **your adapted fork**. The source code contains the Colab T4 native-attention compatibility change. The original model's UniMol branch is disabled in the model arguments; PPI gene features remain enabled. We use a **custom MSE-based objective and custom loop**, not the original paper's complete training protocol.

If `CUDA available` is false, skip model training and inspect the uploaded `results/` files instead.

In [ ]:
import logging, random
from types import SimpleNamespace
import torch.nn.functional as F
from utils import load_dataloader
from models.model_XPert import XPertNet

if not torch.cuda.is_available():
    raise RuntimeError('A supported GPU runtime is required for this training section.')

device = torch.device('cuda:0')
args = SimpleNamespace(
    model='XPert', dataset='l1000_sdst_course_1400',
    drug_feat='unimol', pretrained_mode='global', mode='train',
    include_cell_idx=True, wo_HG=False, wo_atom=False,
    wo_atom_HG=False, wo_unimol=True, wo_ppi=False,
    use_gene_pos_emed=False, output_attention=False,
    output_cls_embed=False
)
logger = logging.getLogger('xpert_course_large')
logger.setLevel(logging.INFO)

train_loader, val_loader, test_loader = load_dataloader(
    args, cfg, logger, nfold='split_course_large'
)
assert (len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)) == (1000,200,200)
print('Train/validation/test:', *(len(x.dataset) for x in
                                 (train_loader, val_loader, test_loader)))

def course_loss(o):
    treated_direct, control_direct, delta, treated, control = o[:5]
    return (F.mse_loss(delta, treated - control)
            + 0.2 * F.mse_loss(treated_direct, treated)
            + 0.1 * F.mse_loss(control_direct, control))

@torch.no_grad()
def validation_mse(net, loader):
    net.eval()
    sq_sum, count = 0.0, 0
    for batch in loader:
        o = net(batch)
        target = o[3]
        prediction = o[4] + o[2]
        sq_sum += ((target - prediction)**2).sum().item()
        count += target.numel()
    return sq_sum / count

## 7 · Train and select by **validation MSE only**
Original run: **10 epochs**, Adam (learning rate `1e-4`, weight decay `1e-5`), batch size 1, gradient clipping at 1.0, one fixed seed (`2026`). Training began from freshly initialized model weights, **not** from the 5-batch probe used earlier.

This section writes the best checkpoint and training history to Google Drive. The upload is durable across Colab runtime resets. If GPU hours are unavailable, there is no need to rerun: the completed experiment's `results/training_history_large.csv` and `results/test_predictions_large.npz` already exist on GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
out_dir = Path('/content/drive/MyDrive/AIDD8122_XPert/large_run')
out_dir.mkdir(parents=True, exist_ok=True)

random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)
torch.cuda.manual_seed_all(2026)

model = XPertNet(args, cfg, device, logger)
model.init_weights()
model = model.to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

history, best_val, best_epoch = [], float('inf'), None
for epoch in range(1, 11):
    model.train()
    epoch_losses = []
    for batch in train_loader:
        optim.zero_grad(set_to_none=True)
        loss = course_loss(model(batch))
        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite training loss at epoch {epoch}')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        epoch_losses.append(loss.item())

    val_mse = validation_mse(model, val_loader)
    train_loss = float(np.mean(epoch_losses))
    history.append({'epoch':epoch, 'train_loss':train_loss, 'val_mse':val_mse})
    pd.DataFrame(history).to_csv(out_dir / 'training_history_large.csv', index=False)
    print(f'Epoch {epoch:02}/10: train loss={train_loss:.6f}, val MSE={val_mse:.6f}', flush=True)
    if val_mse < best_val:
        best_val, best_epoch = val_mse, epoch
        torch.save(model.state_dict(), out_dir / 'xpert_hg_1400_best.pt')
print('Best epoch:', best_epoch, '| Validation MSE:', best_val)

## 8 · Evaluate once on the held-out test set
The test partition remains untouched during optimization and checkpoint selection. Group labels describe drugs **seen/unseen in the supervised training subset**; these are exploratory and **not** a strict cold-drug benchmark. Drug embeddings may carry external information.

Below, the sample order is checked indirectly by comparing the no-change MSE against the previously computed deterministic baseline.

In [ ]:
checkpoint = out_dir / 'xpert_hg_1400_best.pt'
model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
model.eval()

true_list, pred_list, ctl_list = [], [], []
with torch.no_grad():
    for batch in test_loader:
        o = model(batch)
        true_list.append(o[3].cpu().numpy())
        ctl_list.append(o[4].cpu().numpy())
        pred_list.append((o[4] + o[2]).cpu().numpy())
y_true = np.concatenate(true_list)
y_pred = np.concatenate(pred_list)
y_ctl = np.concatenate(ctl_list)
assert y_true.shape == y_pred.shape == y_ctl.shape == (200,978)
assert np.isfinite(y_pred).all()
assert abs(np.mean((y_true-y_ctl)**2)-base_mse) < 1e-4

train_drug_set = set(train_loader.dataset.raw_data.obs['pert_idx'].astype(int))
test_drug_ids = test_loader.dataset.raw_data.obs['pert_idx'].astype(int).to_numpy()
seen_mask = np.isin(test_drug_ids, list(train_drug_set))
assert len(seen_mask) == len(y_true)

def pearson_delta(true, pred, control):
    corr = []
    for t,p,c in zip(true,pred,control):
        a,b=t-c,p-c
        if np.std(a) >= 1e-8 and np.std(b) >= 1e-8:
            corr.append(np.corrcoef(a,b)[0,1])
    return float(np.mean(corr)) if corr else np.nan

def metrics(model_name, group, true, pred, control):
    mse = float(np.mean((true-pred)**2))
    return {'Group':group, 'Model':model_name, 'N':len(true),
            'MSE':mse, 'RMSE':float(np.sqrt(mse)),
            'MAE':float(np.mean(np.abs(true-pred))),
            'Pearson_Delta':pearson_delta(true,pred,control),
            'Mean_Abs_Change':float(np.mean(np.abs(pred-control)))}

rows=[]
for group, mask in [
    ('All',np.ones(len(y_true),dtype=bool)),
    ('Seen in training',seen_mask),
    ('Unseen in training',~seen_mask)
]:
    if mask.any():
        rows.append(metrics('No-change Baseline',group,y_true[mask],y_ctl[mask],y_ctl[mask]))
        rows.append(metrics('XPert-HG-only',group,y_true[mask],y_pred[mask],y_ctl[mask]))
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
comparison.to_csv(out_dir / 'model_comparison_large.csv', index=False)
np.savez_compressed(out_dir / 'test_predictions_large.npz',
                    y_true=y_true, y_pred=y_pred, y_control=y_ctl,
                    drug_ids=test_drug_ids, seen_mask=seen_mask)

## 9 · Per-sample errors, figures and drug-cluster bootstrap
Compute per-sample mean squared errors, the proportion of samples with improved MSE, and a **descriptive** 95% percentile interval via 5,000 drug-ID cluster bootstrap replicates. This accounts for repeated test records per drug, **but not** for full cell/batch dependence or between-training-run variability. Results should not be interpreted as proof of broad superiority.

In [ ]:
import matplotlib.pyplot as plt
history_df = pd.read_csv(out_dir / 'training_history_large.csv')
comparison_df = pd.read_csv(out_dir / 'model_comparison_large.csv')
with np.load(out_dir / 'test_predictions_large.npz') as z:
    yt, yp, yc = z['y_true'], z['y_pred'], z['y_control']
    drug_ids = z['drug_ids']
base_err = np.mean((yt-yc)**2, axis=1)
model_err = np.mean((yt-yp)**2, axis=1)
improvement = base_err - model_err
print('Improved test samples:', int((improvement>0).sum()), '/', len(improvement))
print('Mean MSE reduction:', float(improvement.mean()))
print('Median MSE reduction:', float(np.median(improvement)))

unique_drugs, group_idx = np.unique(drug_ids,return_inverse=True)
group_sum = np.bincount(group_idx, weights=improvement)
group_count = np.bincount(group_idx)
rng = np.random.default_rng(2026)
nb = 5000
indices = rng.integers(0,len(unique_drugs),size=(nb,len(unique_drugs)))
bootstrap_means = group_sum[indices].sum(axis=1)/group_count[indices].sum(axis=1)
ci_low, ci_high = np.percentile(bootstrap_means, [2.5,97.5])
pd.DataFrame([{
    'Mean_MSE_Improvement':float(improvement.mean()),
    'CI_Lower':float(ci_low), 'CI_Upper':float(ci_high),
    'N_Test_Samples':len(improvement),'N_Test_Drugs':len(unique_drugs)
}]).to_csv(out_dir/'bootstrap_results.csv',index=False)
print('Drug-cluster bootstrap 95% CI:', ci_low, ci_high)

# Fig 1: Training and validation trajectories
plt.figure(figsize=(7,4))
plt.plot(history_df['epoch'],history_df['train_loss'],marker='o',label='Training loss')
plt.plot(history_df['epoch'],history_df['val_mse'],marker='s',label='Validation MSE')
plt.xlabel('Epoch');plt.ylabel('Metric');plt.title('XPert-HG-only training');plt.legend()
plt.tight_layout();plt.savefig(out_dir/'Figure1_Training.png',dpi=300);plt.show()

# Fig 2: Group-level test MSE
pivot = comparison_df.pivot(index='Group',columns='Model',values='MSE')
pivot = pivot.reindex(['All','Seen in training','Unseen in training'])
pivot.plot.bar(rot=0,figsize=(8,4.5));plt.ylabel('Test MSE')
plt.tight_layout();plt.savefig(out_dir/'Figure2_Model_Comparison.png',dpi=300);plt.show()

# Fig 3: Paired per-sample MSE
plt.figure(figsize=(6,5));plt.scatter(base_err,model_err,alpha=.6)
lim = float(max(base_err.max(),model_err.max())*1.05)
plt.plot([0,lim],[0,lim],'--');plt.xlabel('No-change MSE');plt.ylabel('XPert-HG-only MSE')
plt.tight_layout();plt.savefig(out_dir/'Figure3_Sample_Errors.png',dpi=300);plt.show()

# Fig 4: Distribution of reductions
plt.figure(figsize=(7,4.5));plt.hist(improvement,bins=25)
plt.axvline(0,linestyle='--');plt.axvline(improvement.mean(),linestyle='-')
plt.xlabel('MSE reduction (baseline − XPert)');plt.ylabel('Number of samples')
plt.tight_layout();plt.savefig(out_dir/'Figure4_Improvement_Distribution.png',dpi=300);plt.show()

# Fig 5: Bootstrap interval
plt.figure(figsize=(7,2.8));plt.errorbar(improvement.mean(),0,
    xerr=[[improvement.mean()-ci_low],[ci_high-improvement.mean()]],fmt='o',capsize=7)
plt.axvline(0,linestyle='--');plt.yticks([]);plt.xlabel('MSE reduction (baseline − XPert)')
plt.tight_layout();plt.savefig(out_dir/'Figure5_Bootstrap_CI.png',dpi=300);plt.show()
print('Saved results and five figures to:', out_dir)

## 10 · Recorded results from the completed course run (not newly executed here)
The original uploaded notebook contains these **actual saved outputs**, also archived in the fork's `results/` folder:

| Metric | No-change baseline | XPert-HG-only |
|---|---:|---:|
| Test MSE | 0.519923 | 0.510133 |
| Test RMSE | 0.721057 | 0.714236 |
| Test MAE | 0.442325 | 0.445866 |
| Mean Pearson Δ | undefined | 0.131860 |
| Mean absolute predicted change | 0 | 0.074675 |

- **10 training epochs completed; best validation epoch: 10; best validation MSE: 0.491045.**
- **136 of 200** test records had lower sample-wise MSE with XPert-HG-only; **64** had higher MSE.
- Test mean MSE reduction **0.009790651** (about **1.88%** relative to baseline).
- Drug-cluster bootstrap 95% percentile interval **[0.007555781, 0.012155199]** from 196 unique test-drug IDs.
- Seen in supervised training: 42 test records; unseen in supervised training: 158.
- Main limitation: Pearson Δ is low and predicted perturbation amplitude remains markedly smaller than the observed amplitude.

**Interpretation:** This is a limited improvement in MSE for a modified HG-only model under one random seed and one subsampled evaluation. It does **not** establish that the full original XPert model was successfully reproduced, nor that drug-response mechanisms were learned reliably. The dataset selection is **not strictly drug-disjoint**; do not call it a formal cold-drug benchmark.

### How to inspect results without GPU
Open `results/model_comparison_large.csv`, `results/bootstrap_results.csv`, `results/training_history_large.csv` and `results/figures/` directly on GitHub. The test predictions used for analysis are archived in `results/test_predictions_large.npz`.

### Attribution
Adapted from [GSanShui/XPert](https://github.com/GSanShui/XPert), with public experimental data from [Zenodo record 17182939](https://zenodo.org/records/17182939). Follow upstream licensing and cite the original paper. The code changes (native attention and simplified HG-only objective) and the experiment are documented as course adaptations.